In [ ]:
from pathlib import Path
import re
import yaml
from sentence_transformers import SentenceTransformer

# ESG lexicon location
LEXICON_PATH = Path("./ESG_Lexicon.yml")

# --------------------
# Helper functions
# --------------------
def clean_term(t: str) -> str:
    t = t.strip()
    t = re.sub(r"\s+", " ", t)
    return t.strip(" ,;:.")


def extract_strings(node):
    out = []
    if isinstance(node, str):
        out.append(node)
    elif isinstance(node, list):
        for x in node:
            out.extend(extract_strings(x))
    elif isinstance(node, dict):
        for v in node.values():
            out.extend(extract_strings(v))
    return out


def build_anchor_text(terms):
    cleaned = [clean_term(t) for t in terms if t]
    unique_terms = list(dict.fromkeys(cleaned))
    return "; ".join(unique_terms) + "."


# --------------------
# Load ESG lexicon
# --------------------
with LEXICON_PATH.open("r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

# Extract Environmental and Governance vocabularies
E_terms = extract_strings(lex["E"])
G_terms = extract_strings(lex["G"])

# Construct anchor texts
E_anchor = build_anchor_text(E_terms)
G_anchor = build_anchor_text(G_terms)

# --------------------
# Generate embeddings
# --------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

E_embedding = model.encode(E_anchor)
G_embedding = model.encode(G_anchor)

In [ ]:
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# --------------------
# Load dataset
# --------------------
PARQUET_FILE = "spy_10k_2015_present.parquet"

df = pl.read_parquet(PARQUET_FILE)

texts = df["text"].to_list()
tickers = df["ticker"].to_list()

# --------------------
# SentenceTransformer model
# --------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed firm disclosures
firm_embeddings = model.encode(texts, normalize_embeddings=True)

# --------------------
# ESG anchors
# --------------------
E_anchor = "climate change; emissions reduction; carbon neutrality; renewable energy; sustainability strategy."
G_anchor = "corporate governance; board oversight; executive compensation; shareholder rights; regulatory compliance."

E_vec = model.encode([E_anchor], normalize_embeddings=True)
G_vec = model.encode([G_anchor], normalize_embeddings=True)

# --------------------
# PCA projection
# --------------------
coords = PCA(n_components=2).fit_transform(firm_embeddings)

# --------------------
# Similarity to anchors
# --------------------
E_scores = cosine_similarity(E_vec, firm_embeddings)[0]
G_scores = cosine_similarity(G_vec, firm_embeddings)[0]

# --------------------
# Environmental scatter
# --------------------
plt.figure(figsize=(8,6))
plt.scatter(coords[:,0], coords[:,1], c="yellow", alpha=0.7)
plt.title("Environmental Disclosure Embedding Space")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

# --------------------
# Governance scatter
# --------------------
plt.figure(figsize=(8,6))
plt.scatter(coords[:,0], coords[:,1], c="blue", alpha=0.7)
plt.title("Governance Disclosure Embedding Space")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()